# 🧹 Phase 1: Data Cleaning & Pre-processing

Based on the initial analysis, this notebook implements a **conservative cleaning strategy**.
We aim to fix structural errors (missing IDs) and optimize data types while preserving as much information as possible (e.g., keeping anime with partial data).

**Key Actions:**
1.  **Impute** missing values where logic allows (e.g., `favorites=0`, `is_rewatching=False`).
2.  **Drop** only structurally corrupted rows (missing Primary/Foreign Keys).
3.  **Extract** useful features (e.g., `year` from `start_date`).
4.  **Optimize** data types (float64 -> float32/16, objects -> categories) to reduce memory usage.
5.  **Save** to optimized Parquet format for the next steps.

In [6]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os
import gc
from pathlib import Path
import pprint

# 🧪 Standard Import (requires 'solution' package installed)
from solution.src.common import PROCESSED_DATA_DIR
from solution.src.data import load_raw, get_raw_path
from solution.src.utils import save_refined, print_fancy_progress


# 📦 Import atomic cleaning functions and utils
from nb_01_cleaning import (
    REMAINING_FILES,
    optimize_dtypes,
    get_memory_usage_mb,
    clean_string_columns,
    standardize_dataset
)

# Ensure processed directory exists
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

print(f"Saving to:    {PROCESSED_DATA_DIR}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Saving to:    C:\Users\chrst\repos\unito-hci-data-analysis-25-26\data\processed


## 1. Characters (`characters.csv`)
- Drop rows with missing `character_mal_id`.
- Fill missing `favorites` with 0.
- Drop `about` column (mostly empty/text heavy text not needed for general stats).

In [7]:
# 1. Load
df_chars = load_raw('characters')

# 2. Clean
initial_rows = len(df_chars)

df_chars, std_metrics = standardize_dataset(df_chars, primary_key='character_id')
rows_after_dedup = len(df_chars)

missing_favs = df_chars['favorites'].isna().sum() if 'favorites' in df_chars.columns else 0

df_chars = df_chars.dropna(subset=['character_id'])

dropped_cols = []

df_chars = df_chars.drop(columns=['about'])
dropped_cols.append('about')

report = {
    "Initial Rows": initial_rows,
    "Rows Dropped (Duplicates)": std_metrics['duplicates_dropped'],
    "Rows Dropped (Missing ID)": rows_after_dedup - len(df_chars),
    "Favorites Imputed": missing_favs,
    "Columns Dropped": dropped_cols,
    "Final Shape": str(df_chars.shape)
}

print("--- Characters Cleaning Report ---")
pprint.pprint(report, sort_dicts=False)

# 3. Optimize & Save
df_chars = optimize_dtypes(df_chars)
# save_refined(df_chars, "characters.csv", format='csv')
save_refined(df_chars, PROCESSED_DATA_DIR / "characters.parquet", format='parquet')

# 4. Capture Valid Character IDs for Referential Integrity
valid_character_ids = set(df_chars['character_id'])
print(f"Captured {len(valid_character_ids)} valid Character IDs.")


Loading characters.csv with PyArrow...
PyArrow load failed (CSV parser got out of sync with chunker. This can mean the data file contains cell values spanning multiple lines; please consider enabling the option 'newlines_in_values'.), falling back to standard engine.
Loading characters.csv with Pandas (Engine: c)...


## 2. Anime Details (`details.csv`)
- **Year**: Derive from `start_date` where missing.
- **Dates**: Convert to datetime objects.
- **Missing Stats**: Keep `score`, `rank` as NaN (preserve "unranked" status).
- **Columns**: Keep `demographics` and others even if they look empty (might have sparse data).

In [ ]:
# 1. Load
df_details = load_raw('details')

# 2. Clean
initial_rows = len(df_details)

df_details, std_metrics = standardize_dataset(df_details, primary_key='anime_id')
rows_after_dedup = len(df_details)

df_details = df_details.dropna(subset=['title'])
rows_after_title = len(df_details)

for col in ['start_date', 'end_date']:
    df_details[col] = pd.to_datetime(df_details[col], errors='coerce')

missing_year_before = df_details['year'].isna().sum() if 'year' in df_details.columns else 0
df_details['year'] = df_details['year'].fillna(df_details['start_date'].dt.year)
missing_year_after = df_details['year'].isna().sum() if 'year' in df_details.columns else 0

cat_cols = ['type', 'status', 'source', 'rating', 'season']
for col in cat_cols:
    df_details[col] = df_details[col].astype('category')

report = {
    "Initial Rows": initial_rows,
    "Rows Dropped (Duplicates)": std_metrics['duplicates_dropped'],
    "Rows Dropped (Missing Title)": rows_after_dedup - rows_after_title,
    "Year Imputed": missing_year_before - missing_year_after,
    "Remaining Missing Years": missing_year_after,
    "Final Shape": str(df_details.shape)
}

print("--- Details Cleaning Report ---")
pprint.pprint(report, sort_dicts=False)

# 3. Optimize & Save
df_details = optimize_dtypes(df_details)
# save_refined(df_details, "details.csv", format='csv')
save_refined(df_details, PROCESSED_DATA_DIR / "details.parquet", format='parquet')

# 4. Capture Valid Anime IDs for Referential Integrity
valid_anime_ids = set(df_details['anime_id'])
print(f"Captured {len(valid_anime_ids)} valid Anime IDs.")


## 3. Anime Stats (`stats.csv`)
- **Votes**: Fill NaN score votes with 0.
- **Percentages**: Fill NaN percentages with 0.
- **Conserve Rows**: Do not drop rows just because they have no votes.

In [ ]:
# 1. Load
df_stats = load_raw('stats')

# 2. Clean with Referential Integrity
initial_rows = len(df_stats)

df_stats, metrics = standardize_dataset(df_stats, primary_key='anime_id')

rows_dropped_orphans = 0

df_stats, rows_dropped_orphans = enforce_referential_integrity(df_stats, 'anime_id', valid_anime_ids)

score_cols = df_stats.filter(like='score_').columns
nan_in_scores = df_stats[score_cols].isna().sum().sum()

report = {
    "Initial Rows": initial_rows,
    "Rows Dropped (Duplicates)": metrics['duplicates_dropped'],
    "Rows Dropped (Orphans)": rows_dropped_orphans,
    "NaN scores filled": nan_in_scores,
    "Final Shape": str(df_stats.shape)
}

print("--- Stats Cleaning Report ---")
pprint.pprint(report, sort_dicts=False)

# 3. Optimize & Save
df_stats = optimize_dtypes(df_stats)
# save_refined(df_stats, "stats.csv", format='csv')
save_refined(df_stats, PROCESSED_DATA_DIR / "stats.parquet", format='parquet')


## 4. User Ratings (`ratings.csv`)
- **Huge File**: Process carefully.
- **Username**: Drop NaN (orphan votes).
- **Rewatching**: Assume `False` (0) if missing.

In [ ]:
chunk_size = 5_000_000
chunks = []
csv_path = get_raw_path('ratings')

total_rows_read = 0
total_dropped_username = 0
total_rewatch_imputed = 0
total_dropped_orphans = 0

print(f"Processing {csv_path} in chunks...")

valid_anime_ids_arg = valid_anime_ids

with pd.read_csv(csv_path, chunksize=chunk_size) as reader:
    for i, chunk in enumerate(reader, 1):
        print_fancy_progress(i, len(reader), f'Loading {csv_path.name}: ')

        # --- Chunk Logic Inlined ---
        metrics = {
            'rows_read': len(chunk),
            'dropped_username': 0,
            'rewatch_imputed': 0,
            'dropped_orphans': 0
        }

        chunk = chunk[(chunk['score'] >= 0) & (chunk['score'] <= 10)]

        metrics['rewatch_imputed'] = int(chunk['is_rewatching'].isna().sum())
        chunk['is_rewatching'] = chunk['is_rewatching'].fillna(0)

        chunk = clean_string_columns(chunk)

        chunk, dropped = enforce_referential_integrity(chunk, 'anime_id', valid_anime_ids_arg)
        metrics['dropped_orphans'] = dropped

        # Optimization
        dtype_map = {
            'anime_id': np.int32,
            'score': np.int8,
            'num_watched_episodes': np.int32,
            'status': 'category'
        }
        final_dtype_map = {k: v for k, v in dtype_map.items() if k in chunk.columns}
        chunk = chunk.astype(final_dtype_map)
        # ---------------------------

        total_rows_read += metrics['rows_read']
        total_dropped_username += metrics['dropped_username']
        total_rewatch_imputed += metrics['rewatch_imputed']
        total_dropped_orphans += metrics.get('dropped_orphans', 0)

        chunks.append(chunk)

print("\nConcatenating processed chunks...")
df_ratings = pd.concat(chunks, ignore_index=True)

print(f"--- Ratings Cleaning Report ---")
print(f"Total Rows Read: {total_rows_read}")
print(f"Rows Dropped (Missing Username): {total_dropped_username}")
print(f"Rows Dropped (Orphan Anime ID): {total_dropped_orphans}")
print(f"Rewatching Flags Imputed: {total_rewatch_imputed}")
print(f"Final Shape: {df_ratings.shape}")

df_ratings = optimize_dtypes(df_ratings)
# save_refined(df_ratings, "ratings.csv", format='csv')
save_refined(df_ratings, PROCESSED_DATA_DIR / "ratings.parquet", format='parquet')

del df_ratings, chunks
_ = gc.collect()


## 5. Other Files
Processing the remaining files with standard optimizations:
- `person_details.csv`
- `profiles.csv`
- `favs.csv`
- `recommendations.csv`
- Relational tables (`*_works.csv`, `*_nicknames.csv`)

In [ ]:
results = []
print(f"Processing {len(REMAINING_FILES)} other files...")

valid_ids_map = {}
valid_ids_map['anime_id'] = valid_anime_ids
# We use 'character_id' because files are renamed to standardize IDs
valid_ids_map['character_id'] = valid_character_ids

for i, dataset_name in enumerate(REMAINING_FILES, 1):
    print(f"Processing {dataset_name}...")
    # 1. Read
    df = load_raw(dataset_name)

    # Stats Capture
    initial_rows = len(df)
    initial_mem = get_memory_usage_mb(df)

    # 2. Clean (Generic Logic Inlined)
    actions = []

    df, metrics = standardize_dataset(df)

    if metrics['duplicates_dropped'] > 0:
        actions.append(f"Deduplicated ({metrics['duplicates_dropped']} rows)")

    for col, valid_ids in valid_ids_map.items():
        df, dropped = enforce_referential_integrity(df, col, valid_ids)
        if dropped > 0:
            actions.append(f"Dropped Orphans on {col} ({dropped} rows)")

    if dataset_name == "person_details":
        df['birthday'] = pd.to_datetime(df['birthday'], errors='coerce')
        actions.append("Date Parsing (birthday)")

    if dataset_name == "profiles":
        for date_col in ['joined']:
            df['joined'] = pd.to_datetime(df['joined'], errors='coerce')
            actions.append(f"Date Parsing ({'joined'})")
    # -----------------------------

    # 3. Optimize
    df = optimize_dtypes(df, verbose=False)

    # Capture Final Stats
    final_rows = len(df)
    final_mem = get_memory_usage_mb(df)

    # 4. Save
    # We use file_path.name to get the filename like 'person_details.csv' then replace extension
    out_name = file_path.name.replace('.csv', '.parquet')
    save_refined(df, PROCESSED_DATA_DIR / out_name, format='parquet')

    results.append({
        'file': dataset_name, 
        'rows_src': initial_rows, 
        'rows_dst': final_rows,
        'mem_src': initial_mem,
        'mem_dst': final_mem,
        'actions': actions
    })

# Print Summary
print(f"\n--- Other Files Cleaning Report ---")
for res in results:
    mem_red = (res['mem_src'] - res['mem_dst']) / res['mem_src'] * 100 if res['mem_src'] > 0 else 0
    print(f"[{res['file']}]")
    print(f"  Rows:   {res['rows_src']} -> {res['rows_dst']}")
    print(f"  Memory: {res['mem_src']:.2f} MB -> {res['mem_dst']:.2f} MB ({mem_red:.1f}% reduction)")
    if res['actions']:
        print(f"  Actions: {', '.join(res['actions'])}")
    print("-" * 30)
print(f"Total files processed: {len(results)}")
